In [2]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
load_dotenv()

True

In [43]:
model = ChatOllama(model='qwen2.5:3b')
# model = ChatOllama(model='gemma3:1b', num_predict=5, temperature=1.0)   #num_predict instead of max_toxens for chatollama
response = model.invoke('Which animal has a hump on its back?')
response.content

'The animal with a hump on its back is a camel.'

In [4]:
conversation = [
    {"role":"system", "content":"You are a helpful assistant that translates English to Hindi."},
    {"role":"user", "content":"Translate: You are a girl."},
    {"role":"assistant", "content":"Tum ek ladki ho."},
    {"role":"user", "content":"Translate: You are not a girl."},
]
response = model.invoke(conversation)
response.content

'तुम एक लड़की नहीं हो।'

In [5]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
conversation = [
    SystemMessage("You are a helpful assistant that translates English to Hindi."),
    HumanMessage("Translate: You are a girl."),
    AIMessage("Tum ek ladki ho."),
    HumanMessage("Translate: You are not a girl.")
]
response = model.invoke(conversation)
response.content

'तुम एक लड़की नहीं हो।'

In [6]:
full = None
for chunk in model.stream("Which animal has a hump on its back?"):
    full = chunk if full is None else full+chunk
    # print(chunk.text, end="|", flush=True)
    print(full.content)

print(full.content_blocks)

The
The animal
The animal with
The animal with a
The animal with a prominent
The animal with a prominent h
The animal with a prominent hump
The animal with a prominent hump on
The animal with a prominent hump on its
The animal with a prominent hump on its back
The animal with a prominent hump on its back is
The animal with a prominent hump on its back is the
The animal with a prominent hump on its back is the camel
The animal with a prominent hump on its back is the camel.
The animal with a prominent hump on its back is the camel. Cam
The animal with a prominent hump on its back is the camel. Camels
The animal with a prominent hump on its back is the camel. Camels have
The animal with a prominent hump on its back is the camel. Camels have a
The animal with a prominent hump on its back is the camel. Camels have a very
The animal with a prominent hump on its back is the camel. Camels have a very distinctive
The animal with a prominent hump on its back is the camel. Camels have a very dis

In [7]:
# async for event in model.astream_events('Hello'):
#     if event['event'] == "on_chat_model_start":
#         print(f'Input: {event['data']['input']}')
#     elif event['event'] == 'on_chat_model_stream':
#         print(f'Token: {event['data']['chunk'].text}')
#     elif event['event'] == 'on_chat_model_end':
#         print(f"Full message: {event['data']['output'].text}")
#     else:
#         pass

In [8]:
# responses = model.batch([
#     "Which animal has a hump on its back?",
#     "Which animal has a really long neck?",
#     "Which animal is the tallest in the world?"
# ])
# for response in responses:
#     print(response)

for response in model.batch_as_completed([
    "Which animal has a hump on its back?",
    "Which animal has a really long neck?",
    "Which animal is the tallest in the world?",
    "Why do parrots have colorful feathers?"
]):
    print(response)

(3, AIMessage(content="Parrots have colorful feathers for several important reasons related to their evolution and survival in the wild:\n\n1. **Attracting Mates**: One of the primary functions of vibrant colors is to attract mates. In many bird species, including parrots, bright plumage can be a sign of genetic quality. Female birds often prefer males with more brightly colored feathers as they are generally healthier and better adapted to their environment.\n\n2. **Camouflage in Different Environments**: Another aspect is camouflage. While most parrots live in forests or dense vegetation where colors might help blend in, some species such as the green ones often found on islands have brighter colors because they need to stand out from other objects like water or trees to be effectively camouflaged.\n\n3. **Warning Signals**: In some cases, bright feathers can act as warning signals. For example, certain types of parrots may use their vibrant plumage to signal to potential predators t

In [9]:
from langchain.tools import tool
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])
result = model_with_tools.invoke("What's the weather like in Pune?")
print(result)
for tool_call in result.tool_calls:
    print(f'Tool: {tool_call["name"]}')
    print(f'Args: {tool_call["args"]}')

content='' additional_kwargs={} response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-14T15:04:03.7777406Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8815161700, 'load_duration': 126031000, 'prompt_eval_count': 158, 'prompt_eval_duration': 6680161200, 'eval_count': 21, 'eval_duration': 1976869800, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'} id='lc_run--019bbd08-e346-7282-99cc-3d70921b227f-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '11696c45-f91d-43d6-9952-d7d56fdba5e7', 'type': 'tool_call'}] usage_metadata={'input_tokens': 158, 'output_tokens': 21, 'total_tokens': 179}
Tool: get_weather
Args: {'location': 'Pune'}


In [10]:
messages = [{'role':'user', 'content':"What's the weather like in Pune?"}]
ai_message = model_with_tools.invoke(messages)
# print(ai_message)
messages.append(ai_message)
for tool_call in ai_message.tool_calls:
    if tool_call['name']=='get_weather':
        tool_result = get_weather.invoke(tool_call)
    # print(tool_result)
    messages.append(tool_result)
final_result = model_with_tools.invoke(messages)
final_result.content

'The weather in Pune is currently sunny.'

In [11]:
response = model_with_tools.invoke("What's the weather like in Pune and Goa?")
print(response.tool_calls)                         # multiple tool calls
results = []
for tool_call in response.tool_calls:
    if tool_call["name"]=='get_weather':
        result = get_weather.invoke(tool_call)
    results.append(result)
    print(result)

[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '8ba490d7-1892-415c-adcc-c58c2da73073', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': 'Goa'}, 'id': '0dca6b97-8430-4e6c-9eee-616059b9750f', 'type': 'tool_call'}]
content="It's sunny in Pune" name='get_weather' tool_call_id='8ba490d7-1892-415c-adcc-c58c2da73073'
content="It's sunny in Goa" name='get_weather' tool_call_id='0dca6b97-8430-4e6c-9eee-616059b9750f'


In [12]:
# for chunk in model_with_tools.stream("What's the weather in Pune and Goa?"):
#     # print(chunk)
#     for tool_chunk in chunk.tool_call_chunks:
#         if name := tool_chunk.get('name'):
#             print(f'Tool: {name}')
#         if id_ := tool_chunk.get('id'):
#             print(f'ID: {id_}')
#         if args := tool_chunk.get('args'):
#             print(f'Args: {args}')

gathered = None
for chunk in model_with_tools.stream("What's the weather in Pune and Goa?"):
    gathered = chunk if gathered is None else gathered+chunk
    print(gathered.tool_calls)

[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '2fa2a3e6-5a58-46da-9046-e4a869ec40e2', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '2fa2a3e6-5a58-46da-9046-e4a869ec40e2', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': 'Goa'}, 'id': '044dfc9c-4584-4b9b-9d4c-5248be9df426', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '2fa2a3e6-5a58-46da-9046-e4a869ec40e2', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': 'Goa'}, 'id': '044dfc9c-4584-4b9b-9d4c-5248be9df426', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '2fa2a3e6-5a58-46da-9046-e4a869ec40e2', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': 'Goa'}, 'id': '044dfc9c-4584-4b9b-9d4c-5248be9df426', 'type': 'tool_call'}]


In [ ]:
# from pydantic import BaseModel, Field
# class Movie(BaseModel):
#     """A movie with details."""
#     title: str = Field(..., description="Title of the movie.")
#     year: int = Field(..., description="The year of release of the movie.")
#     director: str = Field(..., description="The director of the movie.")
#     rating: float = Field(..., description="Rating of the movie on IMDB.")

# from typing_extensions import TypedDict, Annotated
# class MovieDict(TypedDict):
#     "A movie with details"
#     title: Annotated[str, ..., "The title of the movie."]
#     year: Annotated[int, ..., "The year of the release of the movie."]
#     director: Annotated[str, ..., "The director of the movie."]
#     actor: Annotated[str, ..., "The lead actor(s) of the movie."]
#     rating: Annotated[float, ..., "Rating of the movie on IMDB."]             

# both pydantic and typedDict can be used without descriptions in some cases

import json
json_schema = {
    "title": "Movie",
    "description": "Details of the movie",
    "type": "object",
    "properties": {
        "title": {
            'type': 'string', 
            'description': 'The title of the movie'
        },
        'year': {
            'type': 'integer',
            'description': 'The year of release of the movie'
        },
        'director': {
            'type': 'string',
            'description': 'The director of the movie'
        },
        'actor': {
            'type': 'string',
            'description': 'The lead actor(s) of the movie'
        },
        'rating': {
            'type': 'number',
            'description': 'The IMDb rating of the movie'
        }
    },
    'required': ['title', 'year', 'director', 'actor', 'rating']
}
# model_with_structure = model.with_structured_output(MovieDict)
model_with_structure = model.with_structured_output(
    json_schema,
    method='json_schema'
)
result = model_with_structure.invoke('Provide details of the movie Shutter Island')
print(result)

title='Shutter Island' year=1970 director='Herman Roux' rating=6.4


In [ ]:
# works for a reasoning model
# chunk = model.invoke("Why do parrots have colorful feathers?")
# reasoning_steps = [r for r in chunk.content_blocks if r['type']=='reasoning']
# print(" ".join(step['reasoning'] for step in reasoning_steps))

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI
# model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
# search_tool = {
#     "google_search": {}
# }
# model_with_tools = model.bind_tools([search_tool])
# # model_with_tools = ChatGoogleGenerativeAI(model='gemini-2.5-flash', tools=[{"google_search":{}}])
# response = model_with_tools.invoke("What's the weather today in Delhi?")
# response.content_blocks

[{'type': 'text',
  'text': 'The weather in Delhi today, Wednesday, January 14, 2026, is currently clear with a temperature of 44°F (7°C), feeling like 43°F (6°C). The humidity is around 96%.\n\nThroughout the day, Delhi is expected to be sunny, with a 10% chance of rain. The temperature will range between a minimum of 41°F (5°C) and a maximum of 60°F (16°C). The minimum temperature is also reported to be around 5 to 6 degrees Celsius, with the maximum reaching between 17 to 20 degrees Celsius. Wind speeds are around 3.08 to 4.9 km/h. The sun rose at approximately 7:15 AM and is expected to set around 5:44 PM today.'}]

In [ ]:
response = model.invoke('Tell me a joke',
                        config={
                            'run_name': 'joke-generation',
                            'tags': ['humor', 'demo'],
                            'metadata': {'user_id': 'user123'}
                        })
response                                # useful when debugging with langsmith etc...

AIMessage(content="Sure! Here's one for you:\n\nWhy couldn't the bicycle stand up by itself?\n\nBecause it was two-tired!\n\nI hope you found that amusing! Do you have any other questions or would you like to hear another joke?", additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-14T17:53:38.121496Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6254340000, 'load_duration': 181719800, 'prompt_eval_count': 33, 'prompt_eval_duration': 596917100, 'eval_count': 48, 'eval_duration': 5400001300, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019bbda4-2cd1-7df0-9d61-83f6cacf2b2f-0', usage_metadata={'input_tokens': 33, 'output_tokens': 48, 'total_tokens': 81})

In [ ]:
from langchain.chat_models import init_chat_model
model = init_chat_model(model='gemma3:1b',
                        model_provider='ollama',
                   temperature=0.5,
                   configurable_fields={'model', 'temperature', 'max_tokens'},
                   config_prefix='first')                  #useful for chain with multiple models
# model.invoke('Tell me a joke')
model.invoke('Tell me a joke',
             config={
                 'configurable':{
                     'first_model': 'qwen2.5:3b',
                     'first_temperature': 1.0,
                     'first_max_tokens': 100,
                 }
             }
)

AIMessage(content='Sure! Here\'s one for you:\n\nWhy couldn\'t the bicycle stand alone?\n\nBecause it was tipping歪 (tipped). \n\nI apologize if this isn\'t what you were expecting—a more traditional "bicycle" pun is a bit of a cliché, but sometimes these jokes just pop up unexpectedly.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-14T18:15:59.3884397Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7547801900, 'load_duration': 144900600, 'prompt_eval_count': 33, 'prompt_eval_duration': 155041800, 'eval_count': 62, 'eval_duration': 7130665400, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019bbdb8-9f1d-70e3-b535-f849cf18f09c-0', usage_metadata={'input_tokens': 33, 'output_tokens': 62, 'total_tokens': 95})